# Feature Engineering Temporel

Description : Créer des features basées sur le temps : nombre d’événements par fenêtre glissante (5 min, 1 h), écart entre connexions successives, activité hors heures de bureau.

## Importation des librairies

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
chemin_CICIDS = '../Data/cicids_clean.csv'
chemin_UNSW   = '../Data/unsw_clean.csv'
chemin_LOGS   = '../Data/logs_clean.csv'

df_cicids = pd.read_csv(chemin_CICIDS, low_memory=False)
df_unsw   = pd.read_csv(chemin_UNSW,   low_memory=False)
df_logs   = pd.read_csv(chemin_LOGS,   low_memory=False)



## Dataset 1 : CIC-IDS-2017

### On cherche les colonnes qui contiennent des informations temporelles

In [4]:
cols_temps = [c for c in df_cicids.columns if any(k in c.lower() for k in ['time', 'duration', 'timestamp', 'date'])]
print("Colonnes temporelles :")
print(cols_temps)
print()
print(df_cicids[cols_temps].head())

Colonnes temporelles :
['Flow Duration']

   Flow Duration
0              3
1            109
2             52
3             34
4              3


Ce dataset ne contient pas de colonne timestamp directe donc on utilise la colonne Flow Duration pour reconstruire des features temporelles.

### 1.1 Durée de flux en secondes

On convertit Flow Duration qui est en ms en s et ainsi observer les connexions très courte (inférieure à 1s) ou très longue car elles sont souvent suspectes.

In [5]:
df_cicids['duree_secondes'] = df_cicids['Flow Duration'] / 1_000_000 # conversion ms en s
print(df_cicids['duree_secondes'].describe().round(2))
print(df_cicids.groupby('Label')['duree_secondes'].mean().round(2))

count    223080.00
mean         16.44
std          31.66
min           0.00
25%           0.08
50%           1.54
75%           8.96
max         120.00
Name: duree_secondes, dtype: float64
Label
BENIGN    15.73
DDoS      16.96
Name: duree_secondes, dtype: float64


La durée moyenne des flux est presque identique entre les classes avec 15.73s pour le trafic normal et 16.96s pour les attaques DDoS. 

### 1.2 Connexion longue supérieure à 10 secondes


In [7]:
df_cicids['connexion_longue'] = (df_cicids['duree_secondes'] > 10).astype(int)
print(df_cicids['connexion_longue'].value_counts())
print(df_cicids.groupby('Label')['connexion_longue'].mean().round(2))

connexion_longue
0    173905
1     49175
Name: count, dtype: int64
Label
BENIGN    0.21
DDoS      0.23
Name: connexion_longue, dtype: float64


Comme on l'a vu prédédemment on sait que le trafic DDoS a un débit moyen par paquet beaucoup plus important que le trafic BENIGN. C'est pour cela que normalement leur connexion doit être plus longue. Ici l'écart entre classes reste assez faible avec 0.21 pour BENIGN contre 0.23 pour DDoS on peut en conclure que la durée seule n'est pas discriminante sur ce dataset.

### 1.3 Débit temporel (octets par seconde)

On calcule le rapport entre le volume total d'octets et la durée du flux ainsi un débit très élevé en très peu de temps est susceptible d'être une attaque.

In [8]:
total_octets = df_cicids['Total Length of Fwd Packets'] + df_cicids['Total Length of Bwd Packets']
df_cicids['debit_octets_sec'] = total_octets / (df_cicids['duree_secondes'] + 0.001) # 0.001 pour éviter division par zéro
print(df_cicids['debit_octets_sec'].describe().round(2))
print(df_cicids.groupby('Label')['debit_octets_sec'].mean().round(2))

count      223080.00
mean        47479.70
std        159094.93
min             0.00
25%            11.82
50%           924.31
75%         16114.17
max      10752802.58
Name: debit_octets_sec, dtype: float64
Label
BENIGN    51189.52
DDoS      44724.70
Name: debit_octets_sec, dtype: float64


De même ici les valeurs entre classes sont très proches avec 51 189 octets/s pour BENIGN contre 44 724 pour DDoS et avec un écart-type important (159 094) et un maximum à 10 752 802 octets/s.

## Dataset 2 : UNSW-NB15


###  Vérification des colonnes temporelles


In [9]:
cols_temps = [c for c in df_unsw.columns if any(k in c.lower() for k in ['time', 'stime', 'ltime', 'date'])]
print("Colonnes temporelles disponibles :", cols_temps)
print(df_unsw[cols_temps].head())

Colonnes temporelles disponibles : ['Stime', 'Ltime', 'Stime_dt', 'Ltime_dt']
        Stime       Ltime             Stime_dt             Ltime_dt
0  1421927414  1421927414  2015-01-22 11:50:14  2015-01-22 11:50:14
1  1421927414  1421927414  2015-01-22 11:50:14  2015-01-22 11:50:14
2  1421927414  1421927414  2015-01-22 11:50:14  2015-01-22 11:50:14
3  1421927414  1421927414  2015-01-22 11:50:14  2015-01-22 11:50:14
4  1421927414  1421927414  2015-01-22 11:50:14  2015-01-22 11:50:14


In [23]:
#on convertit le timestamp unix en datetime
df_unsw['timestamp'] = pd.to_datetime(df_unsw['Stime'], unit='s')
df_unsw = df_unsw.sort_values('timestamp').reset_index(drop=True) # tri chronologique
print("Période du dataset :")
print(f"Début:{df_unsw['timestamp'].min()}")
print(f"Fin: {df_unsw['timestamp'].max()}")

Période du dataset :
Début: 2015-01-22 11:49:37
Fin: 2015-01-22 19:44:02


### 2.1  Nombre de connexions par fenêtre glissante de 5 minutes par IP

Ici on cherche à savoir pour chaque connexion combien de connexions de la même IP source a été générées dans les 5 minutes précédentes.
Une IP qui génère beaucoup de connexions en peu de temps est potentiellement en train d'attaquer.

In [12]:
df_unsw = df_unsw.set_index('timestamp')
df_unsw['nb_conn_5min'] = df_unsw.groupby('srcip')['srcip'].transform(lambda x: x.rolling('5min').count())
df_unsw = df_unsw.reset_index()
print(df_unsw['nb_conn_5min'].describe().round(2))
print(df_unsw.groupby('Label')['nb_conn_5min'].mean().round(2))

count    700001.00
mean        837.79
std         616.76
min           1.00
25%         613.00
50%         654.00
75%         716.00
max        3996.00
Name: nb_conn_5min, dtype: float64
Label
0    838.67
1    811.12
Name: nb_conn_5min, dtype: float64


On remarque que les valeurs entre classes sont presque identiques avec 838.67 pour le trafic normal contre 811.12 pour les attaques.

### 2.2 Nombre de connexions par fenêtre glissante de 1 heure par IP

In [13]:
df_unsw = df_unsw.set_index('timestamp')
df_unsw['nb_conn_1h'] = df_unsw.groupby('srcip')['srcip'].transform(lambda x: x.rolling('1h').count())
df_unsw = df_unsw.reset_index()
print(df_unsw['nb_conn_1h'].describe().round(2))
print(df_unsw.groupby('Label')['nb_conn_1h'].mean().round(2))

count    700001.00
mean       7941.40
std        2924.19
min           1.00
25%        7447.00
50%        7796.00
75%        8257.00
max       14152.00
Name: nb_conn_1h, dtype: float64
Label
0    8047.65
1    4699.83
Name: nb_conn_1h, dtype: float64


Contrairement à la fenêtre 5min ici on observe ici un écart important entre classes avec 8047 connexions par heure pour le trafic normal contre 4699 pour les attaques. Cela peut s'expliquer par le fait que les IP attaquantes ont une activité plus concentrée et courte dans le temps alors que les IP normales maintiennent une activité régulière sur toute l'heure. 

### 2.3 Écart entre connexions successives par IP

Ici pour chaque connexion on calcule le temps écoulé depuis la connexion précédente de la même IP car un écart très faible inférieur à 1 seconde peut indiquer un comportement anormal.

In [14]:
df_unsw['ecart_connexion'] = df_unsw.groupby('srcip')['timestamp'].transform(lambda x: x.diff().dt.total_seconds())
print(df_unsw['ecart_connexion'].describe().round(2))
print(df_unsw.groupby('Label')['ecart_connexion'].mean().round(2))

count    699961.00
mean          1.06
std          17.93
min           0.00
25%           0.00
50%           0.00
75%           1.00
max        3698.00
Name: ecart_connexion, dtype: float64
Label
0    1.06
1    1.09
Name: ecart_connexion, dtype: float64


Ici les classes sont presque identiques avec 1.06s pour le trafic normal contre 1.09s pour les attaques.

In [15]:
df_unsw['hors_bureau'] = (
    (df_unsw['timestamp'].dt.hour < 8) | # avannt 8h
    (df_unsw['timestamp'].dt.hour >= 18) | #après 18h
    (df_unsw['timestamp'].dt.dayofweek >= 5) #week-end
).astype(int)
print(df_unsw['hors_bureau'].value_counts())
print(df_unsw.groupby('Label')['hors_bureau'].mean().round(2))

hors_bureau
0    577505
1    122496
Name: count, dtype: int64
Label
0    0.18
1    0.00
Name: hors_bureau, dtype: float64


Au final on remarque que on obtient 0.18 pour le trafic normal contre 0.00 pour les attaques. Cela signifie que toutes les attaques ont eu lieu pendant les heures de bureau ce qui est normal car la le dataset ne couvre qu'une seule journée de 11h49 à 19h44.


## Dataset 3 : Cybersecurity Threat Detection Logs



### Vérification des colonnes temporelles disponibles


In [16]:
cols_temps = [c for c in df_logs.columns if any(k in c.lower() for k in ['time', 'date', 'timestamp'])]
print("Colonnes temporelles disponibles :", cols_temps)
print(df_logs[cols_temps].head())

Colonnes temporelles disponibles : ['timestamp']
             timestamp
0  2024-05-01T00:00:00
1  2024-07-18T00:00:00
2  2024-04-07T00:00:00
3  2024-10-26T00:00:00
4  2024-10-31T00:00:00


In [17]:
# Conversion en datetime et tri chronologique
col_ts = cols_temps[0]  # on prend la première colonne temporelle trouvée
df_logs['timestamp'] = pd.to_datetime(df_logs[col_ts])
df_logs = df_logs.sort_values('timestamp').reset_index(drop=True)

print("Période couverte :")
print(f"  Début : {df_logs['timestamp'].min()}")
print(f"  Fin   : {df_logs['timestamp'].max()}")

Période couverte :
  Début : 2024-01-01 00:00:00
  Fin   : 2024-12-30 00:00:00


La colonne timestamp de ce dataset ne prend pas en compte les horaires, toutes les requêtes d'une même journée partagent exactement le même timestamp avec seulement 365 valeurs distinctes sur toute l'année 2024.
Cela rend inutilisables les features temporelles prévues dans la consigne.
On va plutôt travailler sur des featuresà l'échelle journalière avec le nombre de requêtes par jour par IP, et distinction semaine / week-end. 

### Nombre de requêtes par jour par IP


In [20]:
df_logs['nb_req_jour'] = df_logs.groupby(['source_ip', 'timestamp'])['source_ip'].transform('count')
print(df_logs['nb_req_jour'].describe().round(2))
print(df_logs.groupby('threat_label')['nb_req_jour'].mean().round(2))

count    6000000.00
mean          47.50
std            7.04
min           20.00
25%           43.00
50%           47.00
75%           52.00
max           79.00
Name: nb_req_jour, dtype: float64
threat_label
benign        47.35
malicious     49.25
suspicious    49.12
Name: nb_req_jour, dtype: float64


Comme le dataset est synthétique ici les valeurs sont très proches entre classes avec 47.35 pour bénin, 49.25 pour malveillant et 49.12 pour suspicieux et avec une distribution homogène (médiane 47, écart-type 7).

### Jour de la semaine 

In [21]:
df_logs['jour_semaine'] = df_logs['timestamp'].dt.dayofweek
print(df_logs.groupby(['threat_label', 'jour_semaine']).size().unstack().round(2))

jour_semaine       0       1       2       3       4       5       6
threat_label                                                        
benign        802317  785240  786120  786275  785067  786974  785618
malicious      17355   17490   17023   17375   17442   17412   17409
suspicious     52512   51102   51437   51397   51858   51061   51516


De même ici avec une distribution est uniforme sur tous les jours de la semaine pour chaque classe. 


## Sauvegarde des datasets 

In [ ]:
df_cicids.to_csv('../Data/cicids_temporal.csv', index=False)
df_unsw.to_csv('../Data/unsw_temporal.csv',index=False)
df_logs.to_csv('../Data/logs_temporal.csv',index=False)


## Bilan global des features temporelles


Dataset CICIDS2017
Ce dataset n'a pas de vrai timestamp on a seulement la durée de chaque flux. On a donc créé des features simples avec la durée en secondes, flag connexion courte et débit en octets par seconde. Aucune de ces features n'est vraiment discriminante ici car les attaques DDoS durent aussi longtemps que le trafic normal. Les fenêtres glissantes et l'activité hors heures de bureau sont impossibles à calculer sans timestamp réel.

Dataset UNSW-NB15
Dans ce dataset il a un vrai timestamp à la seconde sur 8 heures de capture. On a pu créer les 4 features de la consigne : fenêtre 5min, fenêtre 1h, écart entre connexions et activité hors heures de bureau. La fenêtre d'1 heure est la plus utile avec un écart visible entre les classes.

Dataset Logs
Le timestamp ne contient que la date, sans l'heure. Toutes les connexions d'une même journée ont donc le même timestamp. On s'est adapté en travaillant à l'échelle journalière mais les résultats sont uniformes car le dataset est synthétique.
